# Семинар 11. Проверка нормальности распределения

Ноутбук занятия: материал разбирается вместе с преподавателем, ячейки выполняются сверху вниз.

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-11.ipynb`, например `2MP9-Ivanov-Ivan-seminar-11.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Файлов ноутбук не пишет.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-11-tasks.md` с формулировками и ноутбук `seminar-11-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 11».
- Шпаргалка по функциям: `reference/statistics.md`, раздел «Нормальность». Сегодня сдаётся этап 2 проекта, `project/02-eda.md`. Этап 3 — проверка гипотез — выдаётся на семинаре 12, и первая его гипотеза — нормальность; как выглядит его раздел «Нормальность», показывает раздел 7.

Что сегодня. До сих пор мы описывали данные: средние и медианы, графики, выбросы. С этого семинара мы проверяем гипотезы — утверждения о данных, которые можно опровергнуть числом. Первая гипотеза — нормальность распределения: что такое нормальное распределение и зачем оно нужно; как увидеть отклонение от него на графике; логика проверки гипотезы, общая для семинаров 11–13; три теста нормальности и их ловушки; центральная предельная теорема — почему тесты о среднем работают и без нормальности; преобразования и правило «не нормально — что дальше»; раздел «Нормальность» этапа 3.

## 1. Зачем нормальность и что это такое

Нормальное распределение — колокол: значения скапливаются вокруг среднего и симметрично редеют в обе стороны, а форма колокола целиком задаётся двумя числами — средним и стандартным отклонением. Методы семинаров 12 и 13 — t-тест, доверительный интервал для среднего — выведены в предположении, что данные нормальны. Поэтому нормальность проверяют не ради неё самой, а чтобы выбрать следующий метод.

Подключение то же, что на семинаре 10, и две новые библиотеки статистики: модуль `stats` пакета scipy и функция `lilliefors` пакета statsmodels; обе входят в окружение курса. Данные — водители Брянской области с их ДТП, как на семинарах 9 и 10: стаж без кодов 95, 96 и 99; возраст машины — год ДТП минус год выпуска. Возраст больше 100 лет невозможен — на семинаре 10 так выглядели велосипеды с годом выпуска 1 — и заменяется пропуском через `where`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors

sns.set_theme(style='whitegrid')                                         # оформление графиков по умолчанию
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id', parse_dates=['datetime'])   # ДТП Брянской области
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')   # участники ДТП
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды стажа пропусками
drivers = people[people['role'] == 'Водитель'].merge(dtp, left_on='dtp_id', right_index=True)   # водители с их ДТП
drivers['vehicle_age'] = drivers['datetime'].dt.year - drivers['vehicle_year']   # возраст машины в годах
drivers['vehicle_age'] = drivers['vehicle_age'].where(drivers['vehicle_age'] <= 100)   # невозможный возраст в пропуск
drivers[['experience_clean', 'vehicle_age']].describe().round(2)

Стаж известен у 14 028 водителей, возраст машины — у 14 784. Среднее стажа 15,95 больше медианы 13, у возраста машины 11,51 против 10: оба столбца скошены вправо, как возраст машины на семинаре 10. Самой старой машине 62 года: возраста больше 100 лет среди водителей нет — велосипеды с годом выпуска 1 записаны не водителями, — но правило страхует от ошибок ввода. Начнём со стажа.

Нормальную кривую с теми же средним и отклонением рисуют поверх гистограммы. Чтобы высоты были сравнимы, гистограмма строится в плотностях: `stat='density'` делает площадь всех столбцов равной единице, как площадь под кривой. Высоту нормальной кривой в точке `x` даёт `stats.norm.pdf(x, loc=среднее, scale=отклонение)`; её считают на сетке из 200 точек от наименьшего значения до наибольшего — сетку строит `np.linspace`. Функции scipy по умолчанию пропуски не пропускают, поэтому стаж сразу берётся без них, через `dropna()`. Стаж — целые годы, и гистограмма, как на семинаре 8, строится с `discrete=True`: столбец на каждый год, без пустых промежутков между целыми.

In [ ]:
experience = drivers['experience_clean'].dropna()                        # стаж без пропусков
mean, std = experience.mean(), experience.std()                          # параметры нормальной кривой
print(round(mean, 2), round(std, 2))
grid = np.linspace(experience.min(), experience.max(), 200)              # точки для кривой
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x=experience, stat='density', discrete=True, ax=ax)         # гистограмма плотности
ax.plot(grid, stats.norm.pdf(grid, loc=mean, scale=std), color='red')    # кривая нормального распределения
ax.set_title('Стаж водителей и нормальное распределение')
ax.set_xlabel('Стаж, лет')
ax.set_ylabel('Плотность');

Среднее 15,95 года, стандартное отклонение 12,08. Колокол и гистограмма не совпадают. Горб гистограммы левее вершины кривой и выше неё, а справа хвост тянется к 60–70 годам. Слева гистограмма упирается в ноль, а кривая обрезана: у нормального распределения с такими средним и отклонением заметная часть значений пришлась бы на отрицательный стаж. Пики на 10 и 20 годах похожи на округление: круглый стаж записывают чаще соседнего.

Насколько это далеко от нормального? Первая мерка — правило 68–95: у нормального распределения в пределах одного стандартного отклонения от среднего лежат 68,3 % значений, в пределах двух — 95,4 %. Расстояние каждого значения от среднего — модуль разности, `abs()`; среднее логической маски — доля.

In [ ]:
distance = (experience - mean).abs()                                     # удаление от среднего
print(round((distance <= std).mean(), 3), round((distance <= 2 * std).mean(), 3))   # доли в пределах отклонений

68,5 % и 95,9 % — почти как у нормального распределения. По правилу 68–95 стаж близок к нормальному, а по картинке — нет: правило смотрит на середину распределения и не замечает, с какой стороны хвост. Вторая мерка — два числа формы. Асимметрию, `skew()`, считали на семинаре 10: около нуля у симметричного распределения, больше нуля при длинном хвосте справа, меньше нуля — слева. Эксцесс, `kurt()`, измеряет тяжесть хвостов: около нуля у нормального распределения, больше нуля — хвосты тяжелее нормальных, и далёкие от среднего значения встречаются чаще, меньше нуля — хвосты легче, и далёких значений меньше. У нормального распределения оба числа — нули.

In [ ]:
print(round(experience.skew(), 2), round(experience.kurt(), 2))          # асимметрия и эксцесс

Асимметрия 0,81 — заметный правый хвост; эксцесс −0,02 — по тяжести хвостов стаж как нормальный. Стаж почти нормален в середине, но скошен. Правило раздела: нормальность — не свойство данных «вообще», а условие конкретного метода; при больших выборках её заменяет центральная предельная теорема, о ней раздел 5.

## 2. Глазами: Q–Q график

Гистограмма зависит от бинов, и хвосты на ней плохо видны. Точнее показывает Q–Q график, график квантилей. Значения сортируются по возрастанию, и каждое ставится против квантиля стандартного нормального распределения — со средним 0 и отклонением 1 — с тем же номером: по оси X — на сколько стандартных отклонений от среднего отстояло бы значение с этим номером, будь данные нормальными, по оси Y — само значение. Если данные нормальны, точки ложатся на прямую: её сдвиг — среднее, наклон близок к стандартному отклонению. Рисует его `stats.probplot(x, plot=ax)`: синие точки — значения, красная прямая — нормальное распределение, подобранное по данным; подписи осей scipy ставит по-английски, их заменяют.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
stats.probplot(experience, plot=ax)                                      # Q–Q график стажа
ax.set_title('Q–Q график: стаж водителей')
ax.set_xlabel('Квантили нормального распределения')
ax.set_ylabel('Стаж, лет');

Середина лежит близко к прямой, а концы уходят от неё. Слева точки выстроились полкой на нуле выше прямой: нормальное распределение ждёт там отрицательный стаж, а его нет. Справа точки поднимаются над прямой: правый хвост длиннее нормального. Вместе получается дуга, выгнутая вниз, — так на Q–Q графике выглядит правый хвост. Полки из точек — целые годы: у сотен водителей одно и то же значение стажа.

Как читать Q–Q график: точки на прямой — распределение близко к нормальному; правый конец уходит вверх — правый хвост длиннее нормального; левый конец уходит вниз — длиннее левый хвост; концы загибаются в разные стороны, как буква S, — хвосты тяжелее или легче нормальных. Три столбца рядом: стаж, возраст машины и час ДТП. Час берётся из таблицы ДТП, `dtp`: одно ДТП — одно значение, а в таблице водителей ДТП с двумя машинами попало бы в счёт дважды. Подписи трёх панелей ставит короткий цикл `for` по осям.

In [ ]:
age = drivers['vehicle_age'].dropna()                                    # возраст машины без пропусков
hour = dtp['datetime'].dt.hour                                           # час каждого ДТП
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
stats.probplot(experience, plot=axes[0])                                 # Q–Q график стажа
stats.probplot(age, plot=axes[1])                                        # Q–Q график возраста машины
stats.probplot(hour, plot=axes[2])                                       # Q–Q график часа
axes[0].set_title('Стаж водителя')
axes[1].set_title('Возраст машины')
axes[2].set_title('Час ДТП')
for ax in axes:
    ax.set_xlabel('Квантили нормального распределения')
    ax.set_ylabel('Значения')
fig.tight_layout()

Стаж — дуга, правый хвост. Возраст машины — та же дуга, а правый конец взлетает круче: редкие старые машины. Час ДТП — буква S: середина у прямой, а концы — полки на 0 и 23 часах: шкала обрезана с двух сторон, и самых далёких от среднего значений у неё меньше, чем у нормального распределения, — хвосты легче нормальных. Правее левой полки точки идут ниже прямой: ночные часы растянуты — ДТП ночью мало, — это хвост слева. Ни один из трёх столбцов не нормален, и каждый по-своему.

При запасе времени — те же три столбца в ящиках с усами с семинара 8:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
sns.boxplot(x=experience, ax=axes[0])                                    # ящик стажа
sns.boxplot(x=age, ax=axes[1])                                           # ящик возраста машины
sns.boxplot(x=hour, ax=axes[2])                                          # ящик часа
axes[0].set_xlabel('Стаж, лет')
axes[1].set_xlabel('Возраст машины, лет')
axes[2].set_xlabel('Час ДТП')
fig.tight_layout()

У стажа и возраста машины — точки за правым усом и короткий левый ус: правый хвост. У часа точек за усами нет, медиана ближе к правому краю ящика, а левый ус длиннее правого: ночных ДТП мало, и левая часть растянута. Ящик показывает хвосты и крайние значения, но нормальность по нему не проверить: для этого Q–Q график.

## 3. Логика проверки гипотезы

График показывает отклонение, но не отвечает, велико ли оно: может быть, это случайность выборки? На такой вопрос отвечает проверка статистической гипотезы. Логика одна для всех тестов курса, и семинары 12 и 13 стоят на ней.

1. Нулевая гипотеза — утверждение «ничего особенного»: распределение нормальное. Альтернативная гипотеза — её отрицание: распределение не нормальное.
2. Статистика теста — одно число, которое измеряет, насколько данные отклоняются от нулевой гипотезы. У каждого теста своя статистика.
3. p-значение — вероятность получить такое же или ещё большее отклонение, если нулевая гипотеза верна. Маленькое p значит: будь распределение нормальным, такие данные были бы редкостью.
4. Уровень значимости — порог для p, выбранный до теста; в курсе 0,05.
5. Решение: p меньше 0,05 — нулевую гипотезу отвергаем; p не меньше 0,05 — не отвергаем. Никогда не «принимаем» и не «доказали»: «не отвергаем» значит только, что данных не хватило, чтобы её опровергнуть.
6. Вывод словами для читателя отчёта: «распределение стажа отличается от нормального (тест Шапиро–Уилка, p < 0,001)».

Ошибок две. Отвергнуть верную нулевую гипотезу — ошибка первого рода; её вероятность и есть уровень значимости, 5 %. Не отвергнуть неверную — ошибка второго рода; она тем вероятнее, чем меньше выборка.

Юристу эта логика знакома по презумпции невиновности. Нулевая гипотеза — «невиновен», данные — доказательства, уровень значимости — стандарт доказывания. Обвинительный приговор выносят, только если доказательства убедительны; оправдательный приговор не означает, что невиновность доказана, — доказательств вины не хватило. Осудить невиновного — ошибка первого рода, оправдать виновного — второго.

## 4. Три теста

Тест Шапиро–Уилка — самый чувствительный из тестов нормальности на выборках до нескольких тысяч значений. Его статистика W близка к единице у нормальных данных и тем меньше, чем сильнее отклонение. Правило курса: если значений не больше 5 000, тест Шапиро–Уилка идёт по всем; если больше — тест Д’Агостино по всем значениям или тест Шапиро–Уилка на случайной выборке из 500 значений с `random_state=1`, чтобы у всех получались одни и те же числа. Откуда граница 5 000 — через две ячейки. Стаж известен у 14 028 водителей — больше 5 000, поэтому выборка; её даёт метод `sample`.

In [ ]:
sample = experience.sample(500, random_state=1)                          # случайные 500 водителей
result = stats.shapiro(sample)                                           # тест Шапиро–Уилка
result

Результат — объект `ShapiroResult` с двумя полями: `statistic` — это W, `pvalue` — p-значение. Поля достают через точку, а решение записывают словами: сравнение с уровнем значимости внутри условного выражения `'отвергаем' if условие else 'не отвергаем'`.

In [ ]:
print(round(result.statistic, 3), result.pvalue)
'отвергаем' if result.pvalue < 0.05 else 'не отвергаем'                  # решение при уровне 0,05

W 0,938, p 1,4·10⁻¹³ — меньше 0,05: нулевую гипотезу о нормальности отвергаем. Вывод словами: распределение стажа водителей отличается от нормального (тест Шапиро–Уилка, p < 0,001).

Ловушка без ошибки — тот же тест по всей таблице:

In [ ]:
stats.shapiro(experience)                                                # тест по всем значениям

scipy выполнил тест, но предупредил: `For N > 5000, computed p-value may not be accurate` — при n больше 5 000 p-значение может быть неточным. Отсюда и граница правила: тест Шапиро–Уилка рассчитан на выборки до 5 000 значений, а на 14 028 значениях он выдал p 4,4·10⁻⁶² и к выводу ничего не добавил. Поэтому на таблице больше 5 000 значений — выборка 500 или другой тест. Тест Д’Агостино, `stats.normaltest`, соединяет в одну статистику отклонения асимметрии и эксцесса от нулей, которые были бы у нормального распределения, и рассчитан на большие выборки: он идёт по всем значениям.

In [ ]:
stats.normaltest(experience)                                             # тест Д’Агостино

Статистика 1 207, p 6,5·10⁻²⁶³: после запятой 262 нуля. Отвергаем: асимметрия 0,81 на 14 028 значениях случайностью быть не может. Результат — объект `NormaltestResult` с теми же полями `statistic` и `pvalue`.

Третий тест — тест Лиллиефорса из пакета statsmodels. Это вариант теста Колмогорова–Смирнова: тот сравнивает данные с заранее заданным распределением, а Лиллиефорс — с нормальным, у которого среднее и отклонение оценены по самим данным. Функция возвращает не объект с полями, а обычный кортеж из двух чисел: статистику D и p-значение.

In [ ]:
lilliefors(experience)                                                   # статистика D и p

D 0,111, p 0,001. Это p не точное: statsmodels берёт его из таблицы, у которой нижняя граница 0,001, и меньше не покажет. Отвергаем.

Ловушка без ошибки — тест Колмогорова–Смирнова из scipy без параметров:

In [ ]:
stats.kstest(experience, 'norm')                                         # сравнение со стандартным нормальным

p 0,0 — так тест отверг бы что угодно. `stats.kstest(x, 'norm')` сравнивает данные со стандартным нормальным распределением: среднее 0, отклонение 1. Стаж со средним 16 лет с ним, конечно, не совпадает, как не совпал бы любой столбец в годах, рублях или штуках. Для проверки нормальности, когда среднее и отклонение оцениваются по тем же данным, нужен тест Лиллиефорса.

Ещё одна ловушка без ошибки — пропуски. До сих пор тесты получали `experience`, стаж после `dropna()`. Что будет со столбцом таблицы как есть:

In [ ]:
gaps = drivers['experience_clean']                                       # стаж с пропусками
print(stats.shapiro(gaps.sample(500, random_state=1)))                   # тест по выборке с пропусками
fig, ax = plt.subplots(figsize=(5, 5))
stats.probplot(gaps, plot=ax)                                            # Q–Q график с пропусками
ax.set_title('Q–Q график по столбцу с пропусками');

Ни одной ошибки — и ни одного ответа. Шапиро–Уилк вернул `nan`: среди 500 значений выборки есть пропуски. Q–Q график нарисован без прямой, а точки обрываются, не дойдя до правого края: пропуски заняли места в конце ряда, и прямая посчитана как `nan`. Методы pandas пропускают пропуски сами, функции scipy — нет. Поэтому перед любым тестом scipy — `dropna()`.

Выборка 500 нужна только таблицам больше 5 000 значений. Стаж водителей в ДТП во время снегопада, столбец `weather`, — группа поменьше; что будет, если по привычке взять выборку и из неё, — ошибка в следующей ячейке намеренная.

In [ ]:
snow = drivers.loc[drivers['weather'] == 'Снегопад', 'experience_clean'].dropna()   # стаж в снегопад
stats.shapiro(snow.sample(500, random_state=1))                          # тест на выборке 500

`ValueError: Cannot take a larger sample than population when 'replace=False'`: в группе меньше 500 значений, и выборку без возвращения такого размера из неё не взять. Да она и не нужна: значений не больше 5 000, и по правилу курса тест Шапиро–Уилка идёт по всем.

In [ ]:
print(len(snow))
stats.shapiro(snow)                                                      # тест по всей группе

392 значения; W 0,927, p 6,9·10⁻¹³ — отвергаем. Нескольких сотен значений хватает, чтобы тест увидел отклонение. А тридцати?

In [ ]:
stats.shapiro(experience.sample(30, random_state=1))                     # тест на 30 водителях

W 0,950, p 0,167 — больше 0,05: не отвергаем. Но стаж от этого не стал нормальным: это те же водители, по которым тест на 500 значениях нормальность отверг. На малой выборке тест слаб и отклонения не видит — это и есть ошибка второго рода. «Не отвергаем» не значит «нормально»: значит только, что данных мало для вывода. И обратное: на больших выборках тест отвергает нормальность даже при отклонении, которое на графике едва заметно. Поэтому решение принимают по тесту вместе с Q–Q графиком и размером выборки.

## 5. Центральная предельная теорема

Стаж не нормален. Значит ли это, что t-тест семинара 12, который проверяет гипотезы о среднем, к стажу неприменим? Нет — благодаря центральной предельной теореме, ЦПТ. Она говорит: среднее большой выборки распределено почти нормально, каким бы ни было распределение самих значений, а разброс средних равен σ/√n — стандартному отклонению значений, делённому на корень из размера выборки.

Проверим опытом: 1 000 раз возьмём по 100 случайных водителей и посчитаем средний стаж каждой сотни. Генератор `np.random.default_rng(1)` знаком по семинару 3: `rng.choice(values, size=(1000, 100))` вытягивает таблицу 1 000 × 100 случайных значений с возвращением, а `mean(axis=1)` даёт среднее каждой строки. Асимметрию массива NumPy считает Series, поэтому средние на время оборачиваются в `pd.Series`.

In [ ]:
rng = np.random.default_rng(1)                                           # генератор с зерном 1
means = rng.choice(experience.to_numpy(), size=(1000, 100)).mean(axis=1)   # 1000 средних по 100
print(round(pd.Series(means).skew(), 2), round(stats.shapiro(means).pvalue, 3))   # асимметрия и p средних
print(round(means.std(), 3), round(std / np.sqrt(100), 3))               # разброс средних и σ/√n

Асимметрия средних 0,05 — почти ноль, хотя у самого стажа 0,81. Тест Шапиро–Уилка по всем 1 000 средним: p 0,554 — не отвергаем; средних не больше 5 000, и по правилу курса выборка им не нужна. Разброс средних 1,227 почти совпадает с σ/√n = 12,08 / 10 = 1,208. На гистограмме средних — колокол, и нормальная кривая со средним стажа и отклонением σ/√n ложится на неё:

In [ ]:
grid_means = np.linspace(means.min(), means.max(), 200)                  # точки для кривой
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x=means, stat='density', ax=ax)                             # гистограмма средних
ax.plot(grid_means, stats.norm.pdf(grid_means, loc=mean, scale=std / np.sqrt(100)), color='red')   # кривая по ЦПТ
ax.set_title('Средний стаж 1 000 выборок по 100 водителей')
ax.set_xlabel('Средний стаж выборки, лет')
ax.set_ylabel('Плотность');

Вывод для семинара 12: тесты о среднем работают на больших выборках и без нормальности самих значений — нормально распределено среднее. Обычно большой считают выборку от 30 значений; при сильной асимметрии нужно больше.

## 6. Преобразования и что дальше

Иногда нужна симметрия сама по себе: на гистограмме скошенного распределения весь горб прижат к левому краю, и подробности не видны. Правый хвост поджимают преобразованием: корень и логарифм сжимают большие значения сильнее, чем малые. Возраст машины бывает нулевым, а логарифм нуля — минус бесконечность, поэтому берут `np.log1p(x)` — логарифм от x + 1. Асимметрия возраста машины до и после корня и тест Шапиро–Уилка после корня; значений 14 784 — больше 5 000, поэтому тест на выборке 500:

In [ ]:
root = np.sqrt(age)                                                      # корень из возраста
print(round(age.skew(), 2), round(root.skew(), 2))                       # асимметрия до и после
stats.shapiro(root.sample(500, random_state=1))                          # тест после корня

Асимметрия 0,92 после корня стала −0,07 — почти симметрично. Но тест Шапиро–Уилка всё равно отвергает: W 0,992, p 0,010 — меньше 0,05. На 500 значениях тест видит и небольшое отклонение, а возраст машины — целые годы, и полки одинаковых значений никуда не делись. Преобразование не делает данные нормальными «для теста». Логарифм и три гистограммы рядом:

In [ ]:
logged = np.log1p(age)                                                   # логарифм возраста плюс один
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(x=age, discrete=True, ax=axes[0])                           # возраст как есть
sns.histplot(x=logged, bins=20, ax=axes[1])                              # после логарифма
sns.histplot(x=root, bins=20, ax=axes[2])                                # после корня
axes[0].set_title(f'Как есть: асимметрия {age.skew():.2f}')
axes[1].set_title(f'log1p: асимметрия {logged.skew():.2f}')
axes[2].set_title(f'Корень: асимметрия {root.skew():.2f}')
for ax in axes:
    ax.set_xlabel('Значение')
    ax.set_ylabel('Число водителей')
fig.tight_layout()

Логарифм поджимает слишком сильно: асимметрия −0,76, хвост переехал влево — малые возрасты, 0 и 1 год, растянулись сильнее всего. Корень для возраста машины подходит лучше.

Что делать после проверки нормальности — таблица решений для этапа 3 и семинаров 12–13. Сначала смотрят на размер выборки, потом на решение теста:

| Ситуация | Что дальше |
|---|---|
| значений не меньше 30 | t-тест, отвергнута нормальность или нет: по ЦПТ среднее распределено почти нормально |
| значений меньше 30, нормальность отвергнута | непараметрический тест: знаковый тест для одной выборки, семинар 12; Манна–Уитни для двух групп, семинар 13 |
| значений меньше 30, нормальность не отвергнута | мало данных: решать по Q–Q графику и асимметрии |
| нужна симметрия для графика | преобразование — корень или `log1p`; в отчёте сказано, какое и зачем |

Строка «мало данных» продолжает урок раздела 4: уже на 30 значениях тест не увидел отклонения, а на меньшем числе он ещё слабее, и «не отвергаем» почти ничего не значит. Решают глазами: точки Q–Q графика у прямой и асимметрия около нуля — t-тест; дуга или заметная асимметрия — непараметрический тест. Непараметрические тесты сравнивают не средние, а положение значений — сколько их выше заданного числа, какая из двух групп чаще больше, — и длинные хвосты им не мешают.

Правило: преобразуют ради графика, а не ради того, чтобы тест «прошёл».

## 7. Раздел «Нормальность» этапа 3

Этап 3 проекта выдаётся на семинаре 12 и начинается с нормальности: прежде чем проверять гипотезы о средних, нужно решить, какой тест для них годится. Раздел «Нормальность» — таблица по числовым столбцам и Q–Q графики к ней. Строка таблицы: столбец, n, асимметрия, тест, p, решение, что дальше. Образец по трём столбцам Брянской области: тест Д’Агостино по всем значениям, решение при 0,05, «что дальше» — по таблице раздела 6. `np.where` с семинара 3 выбирает одно из двух слов по маске; в таблице раздела 6 два шага, и `np.where` два: первый выбирает слово для малой выборки по решению теста, второй — t-тест или это слово по размеру выборки.

In [ ]:
three = {'стаж водителя': experience, 'возраст машины': age, 'час ДТП': hour}   # три столбца
table = pd.DataFrame({'n': [values.count() for values in three.values()],
                      'асимметрия': [round(values.skew(), 2) for values in three.values()],
                      'тест': 'Д’Агостино',
                      'p': [stats.normaltest(values).pvalue for values in three.values()]}, index=list(three))
table['решение'] = np.where(table['p'] < 0.05, 'отвергаем', 'не отвергаем')   # решение при 0,05
small = np.where(table['решение'] == 'отвергаем', 'непараметрический тест',
                 'мало данных: решать по Q–Q графику и асимметрии')        # если значений меньше 30
table['что дальше'] = np.where(table['n'] >= 30, 't-тест', small)        # сначала размер выборки
table

Все три столбца — «отвергаем», и для всех трёх «что дальше» — t-тест: значений тысячи. p возраста машины выведено нулём: оно меньше самого малого положительного числа, которое хранит компьютер. В отчёте такие p записывают как «p < 0,001».

Таблица считает по правилу, а решает аналитик. Для часа правило выдало t-тест, но средний час — плохая мера: шкала часа циклическая, за 23 часами снова идёт 0, и ДТП в 23 часа и в час ночи в среднем дали бы полдень. Час сравнивают долями — например, долей ночных ДТП в двух группах; тесты для долей — на семинарах 12 и 13. В таблице нормальности час остаётся: она описывает форму распределения, а не выбирает тест для часа.

К таблице в отчёте добавляют Q–Q графики, как в разделе 2, и вывод словами: «Распределения стажа водителей и возраста машины отличаются от нормального (тест Д’Агостино, p < 0,001), но выборки большие, поэтому для сравнения средних годится t-тест. Распределение часа ДТП тоже отличается от нормального; час сравнивают долями, а не средними: шкала циклическая».

Необязательная задача сегодняшних упражнений — такой раздел для своего датасета: таблица, Q–Q графики и вывод «какой тест дальше»; он сдаётся вместе с этапом 3. Напоминание: этап 2 сдаётся сегодня, этап 3 выдаётся на семинаре 12.

## Итоги

Нормальное распределение — колокол, заданный средним и стандартным отклонением. Гистограмма в плотностях с кривой `stats.norm.pdf` сравнивает с ним данные, правило 68–95 проверяет середину, асимметрия `skew()` и эксцесс `kurt()` — форму: у нормального распределения оба числа нули. Q–Q график `stats.probplot(x, plot=ax)`: точки на прямой — близко к нормальному, дуга — хвост, буква S — хвосты тяжелее или легче нормальных. Проверка гипотезы: нулевая и альтернативная гипотезы, статистика, p-значение, уровень значимости 0,05, решение «отвергаем» или «не отвергаем», вывод словами; «не отвергаем» не значит «доказали». Три теста: Шапиро–Уилк `stats.shapiro` — по всем значениям, если их не больше 5 000; на большей таблице scipy предупреждает о неточности p, и тогда берут тест Д’Агостино по всем значениям или Шапиро–Уилка на выборке 500 с `random_state=1`; Д’Агостино `stats.normaltest` — по всем значениям; Лиллиефорс `lilliefors` из statsmodels — вместо `stats.kstest(x, 'norm')`, который сравнивает со стандартным нормальным распределением. Пропуски функции scipy не пропускают: `nan` вместо ответа и Q–Q график без прямой, поэтому сначала `dropna()`. Малая выборка не видит отклонения, большая видит любое. Центральная предельная теорема: средние больших выборок распределены почти нормально с разбросом σ/√n, поэтому t-тест работает и на ненормальных данных, если значений от 30. Что дальше: при 30 значениях и больше — t-тест; при меньшем числе — непараметрический тест, если нормальность отвергнута, и решение по Q–Q графику и асимметрии, если нет; час и другие циклические шкалы сравнивают долями, а не средними. Корень и `log1p` поджимают правый хвост, но тест после них всё равно отвергает; преобразуют ради графика, а не ради теста.

Переходите к упражнениям: откройте `exercises/seminar-11-tasks.ipynb`, формулировки в `exercises/seminar-11-tasks.md`; данные там по Московской области. Функции части Б — инструменты раздела «Нормальность» этапа 3: сводка формы распределения, гистограмма с кривой, Q–Q график, Шапиро–Уилк по правилу курса, таблица нормальности, Лиллиефорс, асимметрия после преобразований, средние случайных выборок; в части В из них собираются отчёт «что дальше», панели Q–Q и рисунок центральной предельной теоремы. Необязательная задача — раздел «Нормальность» этапа 3 для своего датасета, сдаётся вместе с этапом 3. Этап 2 сдаётся сегодня, этап 3 выдаётся на семинаре 12.